# Cleaner concept-bound init on **full** DBpedia

Companion to [clean_bound.ipynb](clean_bound.ipynb), which proposed two less-noisy
weightings of the asserted class hierarchy inside the concept-bound init (the
`class_weight_mode` knob on
[`concept_bound_vectors`](../scripts/_dbpedia_compare.py)) and tested them on the
**10 %** DBpedia corpus:

| variant | `class_weight_mode` | what it does |
|---|---|---|
| **decay** | `"decay"` | keep all materialized ancestors, weight class `c` by `0.5**hops` from the leaf (leaf=1, parent=0.5, …) |
| **most-specific** | `"most_specific"` | drop ancestors; use only the most-specific asserted class |

This notebook trains both on the **full 1.16 B-token corpus** (`walks/all_walks.txt`,
133 M walks), each **with** and **without** the per-row norm cap @ 16 that the
[dbpedia_investigate](dbpedia_investigate.ipynb) notebook found unfreezes the hub
rows during finetuning. A clean **2×2** (init mode × norm policy), all on **P2**,
all with the protected finetune LR **0.0025 → 0.0001**:

| run | init mode | norm policy |
|---|---|---|
| `p2_bound_decay` | decay | global rescale (no cap) |
| `p2_bound_specific` | most_specific | global rescale (no cap) |
| `p2_bound_decay_cap16` | decay | per-row cap @ 16 |
| `p2_bound_specific_cap16` | most_specific | per-row cap @ 16 |

Everything else matches the existing full-corpus runs (P2 protograph pretrain,
`target_norm=8`, `rolled` direction tag, skip-gram dim 200, 5+5 epochs, seed 42).
Models + per-epoch accuracies are saved under `output/dbpedia/<run>/`.

**Baselines (apples-to-apples).** All rows below are evaluated under the **same**
`evaluate_models` harness (LogReg, standardize lens, 89 DLCC splits):
- **vanilla** and **p2_bound (uniform, no-cap)** — re-evaluated here from the cached
  full-corpus `dbpedia_compare` `.kv` files, so the lens matches exactly;
- **p2bound_cap16 (uniform)** — the full-corpus uniform-hierarchy run with the *same*
  cap @ 16 + LR 0.0025→0.0001, read from its `results.json`. This is the exact
  uniform counterpart to the two cap16 runs, so the only thing that changes across
  the cap16 trio is the class-weighting.

The question: **does reweighting the hierarchy (decay / most-specific) help over the
uniform base at full scale, under each norm policy?**

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from _dbpedia_investigate import Model, evaluate_models, load_eval_splits, split_meta, summarize  # noqa: E402

OUT = ROOT / "output" / "dbpedia"
REPORT_DIR = ROOT / "notebooks" / "clean_bound_full"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
LENS = "standardize"     # headline lens (matches dbpedia_investigate)
N_JOBS = 12

# (label, source-kind, locator). order = display order.
#   results : output/dbpedia/<name>/results.json  (own standardize+raw evals)
#   ref_kv  : cached .kv re-evaluated here under the same standardize harness
ROWS = [
    ("vanilla (full ref)",            "ref_kv",  "notebooks/dbpedia_compare/vanilla.kv"),
    ("p2_bound (uniform, no-cap)",    "ref_kv",  "notebooks/dbpedia_compare/p2_bound.kv"),
    ("p2_bound_decay",                "results", "p2_bound_decay"),
    ("p2_bound_specific",             "results", "p2_bound_specific"),
    ("p2bound_cap16 (uniform)",       "results", "p2bound_cap16"),
    ("p2_bound_decay_cap16",          "results", "p2_bound_decay_cap16"),
    ("p2_bound_specific_cap16",       "results", "p2_bound_specific_cap16"),
]
print("rows:", [r[0] for r in ROWS])

rows: ['vanilla (full ref)', 'p2_bound (uniform, no-cap)', 'p2_bound_decay', 'p2_bound_specific', 'p2bound_cap16 (uniform)', 'p2_bound_decay_cap16', 'p2_bound_specific_cap16']


In [2]:
# Reference re-eval: load the cached full-corpus .kv files and score them under the
# SAME standardize harness as the new runs (cached to ref_evals.json -> cheap re-run).
REF_CACHE = REPORT_DIR / "ref_evals.json"
ref_evals = json.loads(REF_CACHE.read_text()) if REF_CACHE.is_file() else {}

needed = [(lbl, loc) for lbl, kind, loc in ROWS if kind == "ref_kv" and lbl not in ref_evals]
if needed:
    from gensim.models import KeyedVectors
    splits = load_eval_splits(ROOT / "v1" / "dbpedia", k=5000)
    print(f"re-evaluating {len(needed)} reference embedding(s) under '{LENS}' "
          f"({len(splits)} splits) ...", flush=True)
    for lbl, loc in needed:
        kv = KeyedVectors.load(str(ROOT / loc), mmap="r")
        m = Model(lbl, kv=kv, dim=kv.vectors.shape[1])
        accs = evaluate_models(splits, [m], scaling=LENS, n_jobs=N_JOBS, seed=42)
        ref_evals[lbl] = accs
        s = summarize(accs)
        print(f"  {lbl:32s} all {s['all']:.3f} (normal {s['normal']:.3f} hard {s['hard']:.3f})",
              flush=True)
        del kv, m
    REF_CACHE.write_text(json.dumps(ref_evals, indent=2) + "\n")
else:
    print("reference evals: all cached")

reference evals: all cached


In [3]:
# Load every row into a common shape: per-split accs (final) + init/final summaries.
def load_row(label, kind, loc):
    if kind == "ref_kv":
        accs = ref_evals.get(label)
        if accs is None:
            return None
        s = summarize(accs)
        return dict(label=label, present=True, init=None, final=s,
                    final_accs=accs, status="ref")
    rj = OUT / loc / "results.json"
    if not rj.is_file():
        return dict(label=label, present=False, init=None, final=None,
                    final_accs=None, status="pending")
    r = json.loads(rj.read_text())
    init = r["per_epoch"][0][LENS]
    final = r["per_epoch"][-1][LENS]
    return dict(label=label, present=True, init=init, final=final,
                final_accs=r["final_accs"][LENS], status="ok",
                seconds=r.get("seconds"), norm_policy=r.get("norm_policy"),
                class_weight_mode=r.get("class_weight_mode"))

loaded = [load_row(*r) for r in ROWS]
for d in loaded:
    if d is None:
        continue
    if not d["present"]:
        print(f"  [pending] {d['label']}  (results.json not written yet)")
print("loaded", sum(1 for d in loaded if d and d['present']), "/", len(ROWS), "rows")

loaded 7 / 7 rows


## Headline — mean LogReg test accuracy (standardize lens, all DLCC splits)

`init_all` is epoch 0 (the init before any finetuning); `final_all` is epoch 5.
`normal` / `hard` are the final means over the easy / `_hard` split halves.
Best `final_*` per column in **bold green**. (Pending runs are omitted until their
`results.json` is written.)

In [4]:
def headline_row(d):
    if d is None or not d["present"]:
        return None
    f = d["final"]; i = d["init"]
    return {
        "init_all":   (i["all"] if i else np.nan),
        "final_all":  f["all"],
        "normal":     f["normal"],
        "hard":       f["hard"],
        "card_H":     f["cardinality (tc09-12) H"],
        "indiv_H":    f["individual {e} (tc04-06) H"],
    }

hl = {d["label"]: headline_row(d) for d in loaded if d and d["present"]}
df_hl = pd.DataFrame(hl).T[["init_all", "final_all", "normal", "hard", "card_H", "indiv_H"]]
display(df_hl.round(4).style.highlight_max(
    subset=["final_all", "normal", "hard", "card_H", "indiv_H"], axis=0,
    props="font-weight:bold;color:#1a7f37;"))

,init_all,final_all,normal,hard,card_H,indiv_H
vanilla (full ref),nan,0.823000,0.893700,0.690700,0.687100,0.869000
"p2_bound (uniform, no-cap)",nan,0.807800,0.856300,0.717200,0.711200,0.773500
p2_bound_decay,0.782300,0.810700,0.862200,0.714300,0.708400,0.772100
p2_bound_specific,0.778000,0.806500,0.853300,0.718900,0.711600,0.782700
p2bound_cap16 (uniform),0.784900,0.850300,0.911200,0.736500,0.720100,0.892000
p2_bound_decay_cap16,0.789600,0.859900,0.915300,0.756200,0.738400,0.895800
p2_bound_specific_cap16,0.783900,0.848800,0.912500,0.729700,0.723300,0.902500


## Category breakdown (final epoch, standardize lens)

DLCC families, normal (`N`) and hard (`H`) means: existence (tc01-03),
individual `{e}` (tc04-06), qualified-existence (tc07-08), cardinality (tc09-12).

In [5]:
FAM_COLS = [
    "existence (tc01-03) N", "existence (tc01-03) H",
    "individual {e} (tc04-06) N", "individual {e} (tc04-06) H",
    "qualified-exist (tc07-08) N", "qualified-exist (tc07-08) H",
    "cardinality (tc09-12) N", "cardinality (tc09-12) H",
]
fam = {d["label"]: {c: d["final"][c] for c in FAM_COLS}
       for d in loaded if d and d["present"]}
df_fam = pd.DataFrame(fam).T[FAM_COLS]
display(df_fam.round(3).style.highlight_max(axis=0, props="font-weight:bold;color:#1a7f37;"))

,existence (tc01-03) N,existence (tc01-03) H,individual {e} (tc04-06) N,individual {e} (tc04-06) H,qualified-exist (tc07-08) N,qualified-exist (tc07-08) H,cardinality (tc09-12) N,cardinality (tc09-12) H
vanilla (full ref),0.918000,0.645000,0.912000,0.869000,0.920000,nan,0.860000,0.687000
"p2_bound (uniform, no-cap)",0.907000,0.721000,0.814000,0.773000,0.907000,nan,0.843000,0.711000
p2_bound_decay,0.865000,0.718000,0.833000,0.772000,0.930000,nan,0.867000,0.708000
p2_bound_specific,0.838000,0.725000,0.817000,0.783000,0.919000,nan,0.874000,0.712000
p2bound_cap16 (uniform),0.939000,0.747000,0.928000,0.892000,0.941000,nan,0.876000,0.720000
p2_bound_decay_cap16,0.942000,0.778000,0.928000,0.896000,0.944000,nan,0.884000,0.738000
p2_bound_specific_cap16,0.946000,0.697000,0.928000,0.902000,0.944000,nan,0.874000,0.723000


## Per-test-case breakdown (final epoch, standardize lens)

Final LogReg test accuracy per test case (mean over domains), one row per tc with a
separate `_hard` row.

In [6]:
def per_tc(accs):
    if accs is None:
        return {}
    b = {}
    for split, a in accs.items():
        tc, _dom, hard = split_meta(split)
        b.setdefault(f"{tc}_hard" if hard else tc, []).append(a)
    return {k: float(np.mean(v)) for k, v in b.items()}

cols = {d["label"]: per_tc(d["final_accs"]) for d in loaded if d and d["present"]}
df_tc = pd.DataFrame(cols)
tcs = sorted({r[:-5] if r.endswith("_hard") else r for r in df_tc.index},
             key=lambda s: int(s[2:]) if s[2:].isdigit() else 99)
order = []
for tc in tcs:
    if tc in df_tc.index: order.append(tc)
    if f"{tc}_hard" in df_tc.index: order.append(f"{tc}_hard")
df_tc = df_tc.loc[order]
df_tc.loc["MEAN"] = df_tc.mean()
display(df_tc.round(3).style.highlight_max(axis=1, props="font-weight:bold;color:#1a7f37;"))

,vanilla (full ref),"p2_bound (uniform, no-cap)",p2_bound_decay,p2_bound_specific,p2bound_cap16 (uniform),p2_bound_decay_cap16,p2_bound_specific_cap16
tc01,0.910000,0.918000,0.918000,0.924000,0.951000,0.959000,0.957000
tc01_hard,0.662000,0.793000,0.778000,0.793000,0.767000,0.783000,0.785000
tc02,0.916000,0.914000,0.806000,0.715000,0.933000,0.928000,0.932000
tc02_hard,0.637000,0.685000,0.688000,0.692000,0.738000,0.775000,0.653000
tc03,0.930000,0.886000,0.872000,0.883000,0.934000,0.937000,0.948000
tc04,0.905000,0.785000,0.809000,0.791000,0.903000,0.902000,0.903000
tc04_hard,0.897000,0.770000,0.777000,0.775000,0.916000,0.915000,0.920000
tc05,0.925000,0.842000,0.859000,0.846000,0.969000,0.969000,0.970000
tc06,0.905000,0.811000,0.825000,0.808000,0.905000,0.905000,0.903000
tc06_hard,0.841000,0.777000,0.767000,0.790000,0.868000,0.877000,0.885000


## Reading the table

Final epoch-5 LogReg test accuracy, standardize lens, 89 DLCC splits (best per column **bold**):

| variant | init | final | normal | hard | card-H | indiv-H |
|---|---|---|---|---|---|---|
| vanilla (full ref) | – | 0.823 | 0.894 | 0.691 | 0.687 | 0.869 |
| p2_bound (uniform, no-cap) | – | 0.808 | 0.856 | 0.717 | 0.711 | 0.773 |
| p2_bound_decay (no cap) | 0.782 | 0.811 | 0.862 | 0.714 | 0.708 | 0.772 |
| p2_bound_specific (no cap) | 0.778 | 0.806 | 0.853 | 0.719 | 0.712 | 0.783 |
| p2bound_cap16 (uniform) | 0.785 | 0.850 | 0.911 | 0.737 | 0.720 | 0.892 |
| **p2_bound_decay_cap16** | 0.790 | **0.860** | **0.915** | **0.756** | **0.738** | 0.896 |
| p2_bound_specific_cap16 | 0.784 | 0.849 | 0.912 | 0.730 | 0.723 | **0.902** |

**1. The norm cap dominates; the weighting is a second-order tweak.** The cleaner
hierarchy weighting moves the *init* by <0.01, and at full corpus the no-cap variants
(decay 0.811, specific 0.806) sit right on the uniform no-cap base (0.808) and **below
vanilla (0.823)** — the classic frozen-hub failure: a single global rescale leaves hub
rows immovable at LR 0.0025, so the `{e}` individual family (tc04–06) that vanilla learns
for free stays stuck (`indiv-H` 0.772 / 0.783 vs vanilla 0.869). Adding the per-row cap
@ 16 unfreezes those hubs and lifts **every** bound variant above vanilla: **+0.049** for
decay (0.811→0.860), **+0.043** for most_specific (0.806→0.849). The cap is worth ~5
points; the weighting scheme ~1.

**2. Best overall: `p2_bound_decay_cap16` (0.860).** Wins all / normal / hard / card-H —
beating **vanilla by +0.037** and the **uniform cap16 base by +0.010**. The gain over
uniform is small in aggregate but concentrated where the cleaner hierarchy should help —
**hard splits (+0.019)** and the **cardinality-hard family (+0.018, 0.738 vs 0.720)** —
mirroring the synthetic `clean_bound` finding that pruning generic-superclass mass helps
the counting tasks. tc09 (cardinality) is the headline: decay_cap16 → 0.939 / 0.898-hard
vs vanilla 0.877 / 0.763.

**3. decay ≳ most_specific.** Under the cap, decay beats most_specific by +0.011 overall,
driven by hard (0.756 vs 0.730) and cardinality. most_specific isn't dominated though: it
has the best `indiv-H` (0.902) and is marginally better on easy existence splits — keeping
only the leaf class sharpens the individual-`{e}` family but discards the inherited-
superclass signal the hard/qualified tasks lean on (tc02_hard: decay 0.775 vs specific
0.653). decay (0.5) is the safer default, matching the synthetic verdict.

**4. Where the bound construction beats vanilla regardless of cap:** the
qualified-existence and cardinality families it was built for — **tc07** (0.985–0.990 vs
vanilla 0.948) and **tc09** (0.90+ vs 0.877, +0.08–0.14 on hard). Even the no-cap variants
win these; they only trail overall because the frozen `{e}` family (tc04–06) drags the
mean down until the cap frees it.

**Bottom line.** On full DBpedia the cleaner weightings behave as on synthetic but the
margins compress: the **cap @ 16 is the decisive ingredient** (the init alone can't beat a
corpus-saturated vanilla while its hubs are frozen), and on top of a capped init,
**decay-weighted hierarchy is the best single recipe (0.860)** — modestly but consistently
ahead of the uniform base, with the edge on the hard and cardinality splits.
